# Elevation API: single point and keyed point list

This compact notebook shows the two basic client patterns without topographic-profile processing. The first request supplies one latitude/longitude pair. The second supplies a table whose user-owned `db_key` values are returned unchanged for joining elevations back to another dataset.

In [1]:
from __future__ import annotations

import json
import re
from io import StringIO
from pathlib import Path

import pandas as pd
import requests
from IPython.display import display

SERVICE_BASE_URL = "https://elevation.logiccloudgeo.com"
REQUEST_TIMEOUT_SECONDS = 90
ELEVATION_UNITS = "feet"

health_response = requests.get(f"{SERVICE_BASE_URL}/health", timeout=15)
health_response.raise_for_status()
display(pd.Series(health_response.json(), name="value").to_frame())

,value
status,ok
service,USGS Elevation Service
version,0.1.0
provider,py3dep


## 1. Request one elevation

Edit the named latitude and longitude values. Named query parameters avoid x/y ordering ambiguity.

In [2]:
single_latitude = 30.4383
single_longitude = -84.2807

single_response = requests.get(
    f"{SERVICE_BASE_URL}/api/v1/elevation",
    params={
        "latitude": single_latitude,
        "longitude": single_longitude,
        "units": ELEVATION_UNITS,
    },
    timeout=REQUEST_TIMEOUT_SECONDS,
)
if not single_response.ok:
    raise RuntimeError(
        f"Single-point request failed with HTTP {single_response.status_code}:\n"
        f"{single_response.text[:1000]}"
    )

single_document = single_response.json()
single_metadata = {key: value for key, value in single_document.items() if key != "result"}
display(pd.Series(single_metadata, name="value").to_frame())
display(pd.DataFrame([single_document["result"]]))

,value
units,feet
horizontal_crs,EPSG:4326
dataset,USGS 3DEP 1/3 arc-second bare-earth DEM
provider,py3dep
approximate_resolution_m,10.0
vertical_reference,NAVD88 over CONUS; source metadata governs oth...


,db_key,latitude,longitude,elevation,status,message
0,None,30.4383,-84.2807,204.786716,success,None


## 2. Request elevations for a keyed point list

Paste rows copied from Excel into `BULK_TABLE_TEXT`, or replace the read statement with `pd.read_csv(...)`. Each `db_key` must be unique and is returned unchanged. This intentionally simple example sends one request and therefore accepts no more than 500 rows. The topographic-profile notebook demonstrates sequential batching for larger tables.

In [3]:
BULK_TABLE_TEXT = """db_key\tlatitude\tlongitude
FL-BRITTON-HILL\t30.9886\t-86.2812
FL-TALLAHASSEE\t30.4383\t-84.2807
FL-GULF-01\t27.0000\t-85.0000
"""

bulk_points = pd.read_csv(StringIO(BULK_TABLE_TEXT.strip()), sep="\t")
display(bulk_points)

,db_key,latitude,longitude
0,FL-BRITTON-HILL,30.9886,-86.2812
1,FL-TALLAHASSEE,30.4383,-84.2807
2,FL-GULF-01,27.0000,-85.0000


In [4]:
required_columns = {"db_key", "latitude", "longitude"}
missing_columns = required_columns - set(bulk_points.columns)
if missing_columns:
    raise ValueError(f"Missing columns: {sorted(missing_columns)}")
if not 1 <= len(bulk_points) <= 500:
    raise ValueError("This simple example requires between 1 and 500 rows.")
if bulk_points["db_key"].duplicated().any():
    raise ValueError("db_key values must be unique.")
key_pattern = re.compile(r"^[A-Za-z0-9][A-Za-z0-9_.-]{0,63}$")
valid_keys = bulk_points["db_key"].astype(str).map(
    lambda value: bool(key_pattern.fullmatch(value))
)
if not valid_keys.all():
    raise ValueError("At least one db_key violates the API identifier rules.")
for column, lower, upper in (
    ("latitude", -90.0, 90.0),
    ("longitude", -180.0, 180.0),
):
    bulk_points[column] = pd.to_numeric(bulk_points[column], errors="raise")
    if not bulk_points[column].between(lower, upper).all():
        raise ValueError(f"{column} contains an out-of-range value.")
print(f"Validated {len(bulk_points):,} keyed points.")

Validated 3 keyed points.


In [5]:
bulk_payload = {
    "units": ELEVATION_UNITS,
    "points": bulk_points[["db_key", "latitude", "longitude"]].to_dict(
        orient="records"
    ),
}
bulk_response = requests.post(
    f"{SERVICE_BASE_URL}/api/v1/elevations",
    json=bulk_payload,
    timeout=REQUEST_TIMEOUT_SECONDS,
)
if not bulk_response.ok:
    try:
        error_text = json.dumps(bulk_response.json(), indent=2)
    except ValueError:
        error_text = bulk_response.text[:1000]
    raise RuntimeError(
        f"Bulk request failed with HTTP {bulk_response.status_code}:\n{error_text}"
    )

bulk_document = bulk_response.json()
bulk_metadata = {key: value for key, value in bulk_document.items() if key != "results"}
bulk_results = pd.DataFrame(bulk_document["results"])
display(pd.Series(bulk_metadata, name="value").to_frame())
display(bulk_results)

,value
units,feet
horizontal_crs,EPSG:4326
dataset,USGS 3DEP 1/3 arc-second bare-earth DEM
provider,py3dep
approximate_resolution_m,10.0
vertical_reference,NAVD88 over CONUS; source metadata governs oth...


,db_key,latitude,longitude,elevation,status,message
0,FL-BRITTON-HILL,30.9886,-86.2812,3.395671e+02,success,None
1,FL-TALLAHASSEE,30.4383,-84.2807,2.047867e+02,success,None
2,FL-GULF-01,27.0000,-85.0000,-3.280837e+06,success,None


## 3. Join results back by `db_key`

The join demonstrates why the user-owned key belongs in the general point-list workflow even though the profile notebook can generate temporary API keys internally.

In [6]:
result_attributes = bulk_results[["db_key", "elevation", "status", "message"]]
joined_results = bulk_points.merge(
    result_attributes,
    on="db_key",
    how="left",
    sort=False,
    validate="one_to_one",
)
display(joined_results)

repository_candidates = (Path.cwd().resolve(), *Path.cwd().resolve().parents)
repository_root = next(
    path for path in repository_candidates if (path / "pyproject.toml").is_file()
)
output_directory = repository_root / "output"
output_directory.mkdir(parents=True, exist_ok=True)
output_path = output_directory / "elevation_query_results.csv"
joined_results.to_csv(output_path, index=False)
print(output_path)

,db_key,latitude,longitude,elevation,status,message
0,FL-BRITTON-HILL,30.9886,-86.2812,3.395671e+02,success,None
1,FL-TALLAHASSEE,30.4383,-84.2807,2.047867e+02,success,None
2,FL-GULF-01,27.0000,-85.0000,-3.280837e+06,success,None


C:\Users\leste\OneDrive - LogicCloud Geo\Documents\Data\Python_Elevation_Service_UsingHyRiver\usgs-elevation-service\output\elevation_query_results.csv
